In [ ]:
# Necesito consolidar 14 archivos de facturación de 2025 y 2026, son aproximadamente 11 millones de datos que no me caben en excel
# Tengo los archivos en formato texto, para lo cual usaré el siguiente codigo con el fin de consolidar, limpiar y filtrar la
# información que necesito.

In [5]:
import pandas as pd
import glob
import os
import csv

# Se define la ruta de inicio ('.') y el patrón para buscar archivos de texto.
# Voy a usar '**/*.txt' para que Python rastree todas las subcarpetas automáticamente.
ruta_carpeta = '.' 
patron_busqueda = os.path.join(ruta_carpeta, "**/*.txt")

# Defino las 17 columnas que me interesan 
columnas_mantener = [
    'movfue', 'movdoc', 'movfec', 'cardetcco', 'cardetcon', 
    'cardetnit', 'procedimiento', 'cardetfec', 'nomarc', 
    'cardetcan', 'cardetvun', 'cardettot', 'nopos', 
    'valfactura', 'movead', 'movcer', 'movres'
]

# glob.glob crea una lista con las rutas de todos los archivos encontrados.
archivos = glob.glob(patron_busqueda, recursive=True)
lista_datos = []

print(f"Iniciando consolidación de {len(archivos)} archivos...")

# El 'for' procesa un archivo a la vez. Si intentáramos abrir los 11 millones
# de registros de un solo golpe, se pega.
for archivo in archivos:
    try:
        # El archivo TXT no dice de qué año es.
        # Por eso, tomamos el nombre de la carpeta (ej. '2025') para etiquetar cada fila.
        anio_etiqueta = os.path.basename(os.path.dirname(archivo))
        nombre_archivo = os.path.basename(archivo)
        registros_archivo = []
        
        # 'with open' abre el archivo y garantiza que se cierre al terminar, liberando memoria.
        # 'utf-8-sig' es el traductor que entiende las tildes y la letra Ñ de Windows.
        with open(archivo, 'r', encoding='utf-8-sig') as f:
            # DictReader lee cada fila como un 'diccionario' (Nombre de columna: Valor).
            # Porque si las columnas cambian de lugar, él las sigue encontrando por nombre.
            lector = csv.DictReader(f, delimiter='|', quoting=csv.QUOTE_NONE)
            lector.fieldnames = [name.strip() for name in lector.fieldnames] # Quita espacios en títulos
            
            for fila in lector:
                # Si a una fila le falta una columna, .get() pone un "0" porque no puedo eliminar registros
                # Esto evita que el código se detenga por errores de formato en el texto original.
                datos_limpios = {col: fila.get(col, "0") for col in columnas_mantener}
                registros_archivo.append(datos_limpios)

        # Convierte los registros acumulados de este archivo en una tabla (DataFrame).
        df_temp = pd.DataFrame(registros_archivo)
        
        # 'valfactura' viene como texto (ej: "1.500,00") python no suma texto,
        # por lo que quito puntos, cambio comas por puntos y lo vuelvo número.
        df_temp['valfactura'] = pd.to_numeric(
            df_temp['valfactura'].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False), 
            errors='coerce'
        ).fillna(0)

        # Agrego el año que salio del nombre de la carpeta para no perder la fecha de origen.
        df_temp['anio_origen'] = anio_etiqueta
        
        lista_datos.append(df_temp)
        print(f"Procesado: {nombre_archivo}")

    except Exception as e:
        # Si un archivo específico falla, el 'except' atrapa el error y permite que el código siga con los demás.
        print(f"Error en archivo {archivo}: {e}")

# PREGUNTAS Y RESPUESTAS DEL TALLER
if lista_datos:
    # Se unen todas las tablas pequeñas de la lista en una sola base de datos maestra.
    consolidado = pd.concat(lista_datos, ignore_index=True)
    
    # Necesito la izquierda del codigo, se crea una columna con los primeros 3 caracteres de 'procedimiento'.
    consolidado['procedimiento_prefijo'] = consolidado['procedimiento'].astype(str).str[:3]
    
    # se convierte la columna 'movfec' a formato de fecha real.
    # Esto permite que Python extraiga el Mes y el Año automáticamente para los filtros.
    consolidado['movfec'] = pd.to_datetime(consolidado['movfec'], dayfirst=True, errors='coerce')
    consolidado['Mes'] = consolidado['movfec'].dt.month
    consolidado['Año'] = consolidado['movfec'].dt.year

    print("\n" + "="*50)
    print("RESULTADOS DEL ANÁLISIS (AGRUPACIONES)")
    print("="*50)
    
    # PREGUNTA 1: Variación del total facturado en febrero 2025 vs 2026.
    # Filtro por mes 2 y se agrupa por Año para sumar los totales globales.
    res_1 = consolidado[consolidado['Mes'] == 2].groupby('Año')['valfactura'].sum().reset_index()
    res_1.columns = ['Año', 'Valor Total Facturado']
    res_1['Variacion'] = res_1['Valor Total Facturado'].diff()
    res_1['Variacion %'] = res_1['Valor Total Facturado'].pct_change() * 100
    print("\n1. Variación Facturación Total Febrero:")
    display(res_1)
    res_1.to_excel("RESPUESTA_1_VARIACION_FEBRERO.xlsx", index=False)

    # PREGUNTA 2: Detalle del concepto 'PAQU' en febrero 2025 y 2026.
    # Busca la palabra 'PAQU' dentro de la columna 'cardetcon'
    # Se incluye 'cardetcon' en el groupby para que se pueda ver el nombre del concepto en la tabla.
    res_2 = consolidado[
        (consolidado['Mes'] == 2) & 
        (consolidado['cardetcon'].str.contains('PAQU', na=False, case=False))
    ].groupby(['Año', 'cardetcon'])['valfactura'].sum().reset_index()
    res_2.columns = ['Año', 'Concepto (cardetcon)', 'Valor Facturado']
    print("\n2. Detalle de Conceptos 'PAQU' en Febrero:")
    display(res_2)
    res_2.to_excel("RESPUESTA_2_CONCEPTO_PAQU.xlsx", index=False)

    # PREGUNTA 3: Facturación de SURA filtrando por NIT 800088702EP.
    # Busca el nit 800088702EP dentro de la columna 'movcer'
    # Se incluye 'movcer' en el groupby para confirmar visualmente que el filtro funcionó.
    res_3 = consolidado[
        (consolidado['Mes'] == 2) & 
        (consolidado['movcer'].str.contains('800088702EP', na=False))
    ].groupby(['Año', 'movcer'])['valfactura'].sum().reset_index()
    res_3.columns = ['Año', 'Responsable (movcer)', 'Valor Facturado']
    print("\n3. Facturación NIT SURA (800088702EP) en Febrero:")
    display(res_3)
    res_3.to_excel("RESPUESTA_3_FACTURACION_NIT.xlsx", index=False)

    print("\n" + "="*50)
    print(f"Proceso finalizado. Total de registros analizados: {len(consolidado):,}")
    print("Los archivos Excel se han generado en la carpeta del proyecto.")
else:
    print("No se encontraron datos para procesar.")

Iniciando consolidación de 14 archivos...
Procesado: DETALLADA ABR.txt
Procesado: DETALLADA AGO.txt
Procesado: DETALLADA DIC.txt
Procesado: DETALLADA ENE.txt
Procesado: DETALLADA FEB.txt
Procesado: DETALLADA JUL.txt
Procesado: DETALLADA JUN.txt
Procesado: DETALLADA MAR.txt
Procesado: DETALLADA MAY.txt
Procesado: DETALLADA NOV.txt
Procesado: DETALLADA OCT.txt
Procesado: DETALLADA SEP.txt
Procesado: DETALLADA ENE.txt
Procesado: DETALLADA FEB.txt


C:\Users\lenovo\AppData\Local\Temp\ipykernel_8068\869540611.py:79: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  consolidado['movfec'] = pd.to_datetime(consolidado['movfec'], dayfirst=True, errors='coerce')



RESULTADOS DEL ANÁLISIS (AGRUPACIONES)

1. Variación Facturación Total Febrero:


,Año,Valor Total Facturado,Variacion,Variacion %
0,2025,55959372553,NaN,NaN
1,2026,74532539032,1.857317e+10,33.190448



2. Detalle de Conceptos 'PAQU' en Febrero:


,Año,Concepto (cardetcon),Valor Facturado
0,2025,PAQU,1168059121
1,2026,PAQU,2244200838



3. Facturación NIT SURA (800088702EP) en Febrero:


,Año,Responsable (movcer),Valor Facturado
0,2025,800088702EP,5471049419
1,2026,800088702EP,9966826750



Proceso finalizado. Total de registros analizados: 11,586,388
Los archivos Excel se han generado en la carpeta del proyecto.
